# Problem 4 — Statement 2: Top Performers & Underperformers (bản đã chỉnh theo rà soát)

**Yêu cầu đề:** chỉ ra nhóm nhân viên có hiệu suất vượt bậc (Top Performers) và nhóm dưới mức kỳ vọng (Underperformers).

**Các điểm đã sửa so với bản cũ (theo file rà soát của nhóm):**
1. Bản cũ dùng 3 chỉ số đầu vào (doanh số, số hóa đơn, doanh thu TB/hóa đơn) nhưng doanh số = số hóa đơn × doanh thu TB, nên điểm tổng hợp bị **đếm trùng**. Bản này chỉ dùng **2 chỉ số độc lập** (số hóa đơn = quy mô, doanh thu TB/hóa đơn = giá trị).
2. Bản cũ chia đúng 25% mỗi nhóm (chia ép). Bản này phân loại theo **ngưỡng điểm z (±1)**, số người mỗi nhóm do dữ liệu quyết định.
3. Bổ sung **kiểm định độ tin cậy** để nói rõ xếp hạng này đáng tin đến đâu.

**Định nghĩa dùng chung với PS1:** hóa đơn thành công = đơn có `order_status = delivered`; doanh thu một dòng hàng = `quantity × unit_price − discount_amount`.

In [1]:
import pandas as pd
import numpy as np

order = pd.read_csv("ORDER.csv")
items = pd.read_csv("ORDER_ITEMS.csv")
emp   = pd.read_csv("SALES_EMPLOYEE.csv")

items["line_revenue"] = items["quantity"] * items["unit_price"] - items["discount_amount"]
rev = items.groupby("order_id")["line_revenue"].sum().rename("revenue").reset_index()

orders = order.merge(rev, on="order_id")
delivered = orders[orders["order_status"] == "delivered"]
print("Tổng đơn:", len(orders), "| đơn thành công (delivered):", len(delivered))

Tổng đơn: 646945 | đơn thành công (delivered): 516716


## 1. KPI theo nhân viên và kiểm tra đếm trùng

In [2]:
kpi = (delivered.groupby("sales_employee_id")
       .agg(invoices=("order_id", "count"), sales=("revenue", "sum"))
       .reset_index())
kpi["avg_revenue"] = kpi["sales"] / kpi["invoices"]

for c in ["sales", "invoices", "avg_revenue"]:
    print(f"CV {c:12s}: {kpi[c].std()/kpi[c].mean()*100:.2f}%")
print()
print("Ma trận tương quan:")
print(kpi[["sales", "invoices", "avg_revenue"]].corr().round(2))

CV sales       : 2.52%
CV invoices    : 1.84%
CV avg_revenue : 1.77%

Ma trận tương quan:
             sales  invoices  avg_revenue
sales         1.00      0.71         0.68
invoices      0.71      1.00        -0.02
avg_revenue   0.68     -0.02         1.00


Doanh số tương quan mạnh với cả hai chỉ số còn lại (0,71 và 0,68) vì nó là tích của chúng, còn **số hóa đơn và doanh thu TB gần như độc lập** (≈ 0). Vì vậy chỉ giữ hai chỉ số độc lập này để chấm điểm.

## 2. Điểm tổng hợp (2 chỉ số độc lập) và phân loại theo ngưỡng

In [3]:
def z(s): return (s - s.mean()) / s.std(ddof=0)

kpi["z_invoices"]    = z(kpi["invoices"])
kpi["z_avg_revenue"] = z(kpi["avg_revenue"])
kpi["score"] = z(kpi["z_invoices"] + kpi["z_avg_revenue"])   # chuẩn hóa lại để độ lệch chuẩn = 1

def classify(s):
    if s >= 1:  return "Top Performer"
    if s <= -1: return "Underperformer"
    return "Đạt kỳ vọng"

kpi["group"] = kpi["score"].apply(classify)
kpi["rank"]  = kpi["score"].rank(ascending=False, method="min").astype(int)

print(kpi["group"].value_counts())
print()
print(kpi.groupby("group")[["invoices", "avg_revenue", "sales"]].mean().round(0))

group
Đạt kỳ vọng       144
Top Performer      29
Underperformer     27
Name: count, dtype: int64

                invoices  avg_revenue       sales
group                                            
Top Performer     2634.0      25077.0  66037247.0
Underperformer    2534.0      23982.0  60774460.0
Đạt kỳ vọng       2583.0      24587.0  63492842.0


## 3. Kiểm tra độ tin cậy: chênh lệch giữa nhân viên là thật hay ngẫu nhiên?

In [4]:
ids, rv = delivered["sales_employee_id"].values, delivered["revenue"].values
codes, _ = pd.factorize(ids)

# (a) Độ tin cậy chia đôi: chia ngẫu nhiên đơn của mỗi người làm 2 nửa, xem xếp hạng 2 nửa có giống nhau không
rng = np.random.default_rng(42)
corrs = []
for _ in range(200):
    half = rng.random(len(rv)) < 0.5
    a = np.bincount(codes[half],  weights=rv[half])
    b = np.bincount(codes[~half], weights=rv[~half])
    corrs.append(np.corrcoef(a, b)[0, 1])
print(f"(a) Tương quan doanh số giữa 2 nửa đơn hàng: trung bình {np.mean(corrs):.3f} (gần 1 = xếp hạng ổn định, gần 0 = nhiễu)")

# (b) Kiểm định hoán vị: gán ngẫu nhiên đơn hàng cho nhân viên rồi xem độ phân tán có lớn như thực tế không
obs = np.bincount(codes, weights=rv)
obs_cv = obs.std(ddof=1) / obs.mean()
null_cv = []
for _ in range(1000):
    s = np.bincount(rng.permutation(codes), weights=rv)
    null_cv.append(s.std(ddof=1) / s.mean())
null_cv = np.array(null_cv)
print(f"(b) CV doanh số thực tế: {obs_cv*100:.2f}% | CV khi gán ngẫu nhiên: {null_cv.mean()*100:.2f}% "
      f"(khoảng 95%: {np.percentile(null_cv,2.5)*100:.2f}%–{np.percentile(null_cv,97.5)*100:.2f}%)")
print(f"    p-value = {(null_cv >= obs_cv).mean():.3f}")

(a) Tương quan doanh số giữa 2 nửa đơn hàng: trung bình -0.052 (gần 1 = xếp hạng ổn định, gần 0 = nhiễu)
(b) CV doanh số thực tế: 2.52% | CV khi gán ngẫu nhiên: 2.57% (khoảng 95%: 2.36%–2.79%)
    p-value = 0.648


**Cách đọc:** nếu nhân viên thật sự khác nhau về năng lực thì hai nửa đơn hàng phải cho xếp hạng giống nhau (tương quan cao) và độ phân tán thực tế phải lớn hơn rõ rệt so với khi gán ngẫu nhiên. Kết quả cho thấy ngược lại.

## 4. So sánh với cách chia cũ (3 chỉ số, chia đúng 25%)

In [5]:
old = kpi.copy()
old["old_score"] = (z(old["sales"]) + z(old["invoices"]) + z(old["avg_revenue"])) / 3
old["old_group"] = pd.qcut(old["old_score"], 4, labels=["Nhóm 4 (thấp nhất)", "Nhóm 3", "Nhóm 2", "Nhóm 1 (cao nhất)"])

rho = old["old_score"].corr(old["score"], method="spearman")
print(f"Tương quan thứ hạng (Spearman) giữa điểm cũ và điểm mới: {rho:.3f}")
print()
print(pd.crosstab(old["old_group"], old["group"]))

Tương quan thứ hạng (Spearman) giữa điểm cũ và điểm mới: 1.000

group               Top Performer  Underperformer  Đạt kỳ vọng
old_group                                                     
Nhóm 4 (thấp nhất)              0              27           23
Nhóm 3                          0               0           50
Nhóm 2                          0               0           50
Nhóm 1 (cao nhất)              29               0           21


## 5. Xuất kết quả

In [6]:
result = kpi.merge(emp[["sales_employee_id", "name"]], on="sales_employee_id", how="left")
result = result[["rank", "sales_employee_id", "name", "invoices", "sales", "avg_revenue",
                 "z_invoices", "z_avg_revenue", "score", "group"]].sort_values("rank")
result.round(3).to_csv("ps2_top_underperformers.csv", index=False, encoding="utf-8-sig")
print("Đã xuất ps2_top_underperformers.csv:", result.shape)
print()
print("Top 5:");    print(result.head(5)[["rank", "sales_employee_id", "name", "invoices", "avg_revenue", "score"]].round(2))
print()
print("Bottom 5:"); print(result.tail(5)[["rank", "sales_employee_id", "name", "invoices", "avg_revenue", "score"]].round(2))

Đã xuất ps2_top_underperformers.csv: (200, 10)

Top 5:
     rank sales_employee_id            name  invoices  avg_revenue  score
142     1           EMP0143   Bùi Minh Quân      2736     24651.70   2.42
162     2           EMP0163  Phạm Thanh Mai      2623     25538.38   2.18
139     3           EMP0140     Lê Hữu Dũng      2637     25345.50   2.07
57      4           EMP0058  Lê Thanh Trang      2635     25351.68   2.05
130     5           EMP0131    Bùi Ngọc Lan      2634     25328.94   2.00

Bottom 5:
     rank sales_employee_id            name  invoices  avg_revenue  score
173   196           EMP0174   Huỳnh Hữu Nam      2487     24128.75  -2.19
158   197           EMP0159   Phạm Văn Bình      2511     23895.04  -2.22
184   198           EMP0185    Phan Hữu Yến      2479     24032.99  -2.47
49    199           EMP0050  Huỳnh Gia Quân      2512     23666.38  -2.58
197   200           EMP0198    Võ Anh Cường      2457     23946.17  -2.95


## Kết luận và khuyến nghị

- **Kết quả phân loại:** 29 Top Performers (điểm z ≥ +1), 27 Underperformers (điểm z ≤ −1), 144 nhân viên đạt kỳ vọng. Số người mỗi nhóm do dữ liệu quyết định, không bị ép 25%.
- **Bỏ đếm trùng không làm đổi thứ hạng** (Spearman ≈ 1,0), vì với độ phân tán nhỏ như vậy doanh số gần như bằng tổng hai chỉ số kia. Thay đổi này giúp cách chấm điểm đúng về mặt phương pháp và dễ giải thích hơn, còn thứ tự xếp hạng vẫn ổn định so với bản cũ.
- **Mức chênh lệch giữa nhân viên rất nhỏ:** hệ số biến thiên doanh số chỉ 2,5%. Nhân viên cao nhất chỉ hơn thấp nhất khoảng 15%.
- **Chênh lệch này không khác gì ngẫu nhiên:** tương quan giữa hai nửa đơn hàng của cùng một nhân viên gần bằng 0 (−0,05), và độ phân tán thực tế nằm trong khoảng của phép gán ngẫu nhiên (p = 0,65). Nghĩa là nhân viên đứng top ở nửa đơn này chưa chắc còn đứng top ở nửa đơn kia.
- **Khuyến nghị:** nhóm Top/Under ở đây nên dùng để theo dõi, chưa nên làm căn cứ duy nhất cho chính sách thưởng/phạt. Nên kết hợp thêm chỉ tiêu thái độ phục vụ (PS4) và chất lượng giao dịch (PS5) trước khi ra quyết định.